# Объединение датасетов all_v2 + input_data

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

## 1. Загрузка данных

In [2]:
all_v2 = pd.read_csv('../data/processed/all_v2_region_fixed.csv')
inp = pd.read_csv('../data/processed/input_data_clean.csv')

print('all_v2:     ', all_v2.shape)
print('input_data: ', inp.shape)
print()
print('all_v2 колонки:    ', list(all_v2.columns))
print('input_data колонки:', list(inp.columns))

all_v2:      (5474552, 13)
input_data:  (11358150, 15)

all_v2 колонки:     ['price', 'date', 'time', 'geo_lat', 'geo_lon', 'building_type', 'level', 'levels', 'rooms', 'area', 'kitchen_area', 'object_type', 'region']
input_data колонки: ['date', 'price', 'level', 'levels', 'rooms', 'area', 'kitchen_area', 'geo_lat', 'geo_lon', 'building_type', 'object_type', 'postal_code', 'street_id', 'id_region', 'house_id']


## 2. Приведение к единой схеме

In [3]:
FINAL_COLS = ['date', 'price', 'level', 'levels', 'rooms', 'area',
              'kitchen_area', 'geo_lat', 'geo_lon', 'object_type', 'region']

# --- all_v2 ---
df_v2 = all_v2.copy()
# object_type: 1 → 0 (вторичка), 11 → 1 (новостройка)
df_v2['object_type'] = df_v2['object_type'].map({1: 0, 11: 1})
# region уже называется region, building_type и time дропаем
df_v2 = df_v2[FINAL_COLS].copy()
df_v2['source'] = 'all_v2'

print('all_v2 после приведения:', df_v2.shape)
print('object_type:', df_v2['object_type'].value_counts().to_dict())

all_v2 после приведения: (5474552, 12)
object_type: {0: 3863479, 1: 1611073}


In [4]:
# --- input_data ---
df_inp = inp.copy()
# object_type: 0 → 0 (вторичка), 2 → 1 (новостройка)
df_inp['object_type'] = df_inp['object_type'].map({0: 0, 2: 1})
# переименовываем id_region → region
df_inp = df_inp.rename(columns={'id_region': 'region'})
# дропаем лишние колонки
df_inp = df_inp[FINAL_COLS].copy()
df_inp['source'] = 'input_data'

print('input_data после приведения:', df_inp.shape)
print('object_type:', df_inp['object_type'].value_counts().to_dict())

input_data после приведения: (11358150, 12)
object_type: {0: 8362230, 1: 2995920}


## 3. Объединение

In [5]:
merged = pd.concat([df_v2, df_inp], ignore_index=True)

print('Итоговый датасет:', merged.shape)
print()
print('Источники:')
print(merged['source'].value_counts())
print()
print('Период дат:')
merged['date'] = pd.to_datetime(merged['date'])
print(merged.groupby('source')['date'].agg(['min', 'max']))

Итоговый датасет: (16832702, 12)

Источники:
source
input_data    11358150
all_v2         5474552
Name: count, dtype: int64

Период дат:
                  min        max
source                          
all_v2     2018-02-19 2021-05-01
input_data 2021-01-01 2021-12-31


## 4. Проверка результата

In [6]:
print('Типы данных:')
print(merged.dtypes)
print()
print('Пропуски:')
print(merged.isnull().sum())
print()
print('object_type (0=вторичка, 1=новостройка):')
print(merged['object_type'].value_counts())
print()
print('Уникальных регионов:', merged['region'].nunique())
print('Диапазон region:', merged['region'].min(), '—', merged['region'].max())

Типы данных:
date            datetime64[us]
price                    int64
level                    int64
levels                   int64
rooms                    int64
area                   float64
kitchen_area           float64
geo_lat                float64
geo_lon                float64
object_type              int64
region                 float64
source                     str
dtype: object

Пропуски:
date            0
price           0
level           0
levels          0
rooms           0
area            0
kitchen_area    0
geo_lat         0
geo_lon         0
object_type     0
region          0
source          0
dtype: int64

object_type (0=вторичка, 1=новостройка):
object_type
0    12225709
1     4606993
Name: count, dtype: int64

Уникальных регионов: 85
Диапазон region: 1.0 — 92.0


In [7]:
print('Статистика числовых признаков:')
display(merged[['price', 'area', 'kitchen_area', 'level', 'levels', 'rooms']].describe().round(2))

Статистика числовых признаков:


,price,area,kitchen_area,level,levels,rooms
count,1.683270e+07,16832702.00,16832702.00,16832702.00,16832702.00,16832702.00
mean,6.017586e+06,53.37,1.65,6.36,11.64,1.72
std,1.628752e+08,29.30,27.91,5.18,7.01,1.13
min,-2.144967e+09,0.07,-100.00,0.00,0.00,-2.00
25%,2.350000e+06,37.00,3.00,2.00,5.00,1.00
50%,3.650000e+06,47.00,8.00,5.00,10.00,2.00
75%,5.971121e+06,63.00,11.50,9.00,17.00,2.00
max,6.355524e+11,7856.00,9999.00,50.00,50.00,10.00


## 5. Сохранение

In [8]:
output_path = '../data/processed/merged.csv'
merged.to_csv(output_path, index=False)
print(f'Сохранено: {output_path}')
print(f'Итог: {len(merged):,} строк, {merged.shape[1]} колонок')

Сохранено: ../data/processed/merged.csv
Итог: 16,832,702 строк, 12 колонок
